### In this notebook iam doing Medallion Architecture operations for the Dimentional table Products

In [0]:
from pyspark.sql.functions import * 
from delta.tables import DeltaTable


In [0]:
%run /Workspace/Users/puchimanidan5777@gmail.com/FMCG/Set_ups/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","products","Source")

In [0]:
catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")
data_path=f's3://bucket-for-fmcg-project/{data_source}/*.csv'

In [0]:
print(data_path)

s3://bucket-for-fmcg-project/products/*.csv


In [0]:
#Reading data from aws S3
df_source=spark.read.format("csv")\
    .option("inferSchema",True)\
    .option("header",True)\
    .load(data_path)\
    .withColumn("read_timestamp",current_timestamp())\
    .select("*","_metadata.file_name","_metadata.file_size")

In [0]:
display(df_source.limit(10))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-07-03T14:48:50.947Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-07-03T14:48:50.947Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-07-03T14:48:50.947Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-07-03T14:48:50.947Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-07-03T14:48:50.947Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-07-03T14:48:50.947Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-07-03T14:48:50.947Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-07-03T14:48:50.947Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-07-03T14:48:50.947Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-07-03T14:48:50.947Z,products.csv,1388


### Bronze Schema

In [0]:
df_source.write\
    .format('delta')\
    .mode('overwrite')\
    .option("enableChangeDataFeed",True)\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

#Silver Processing

In [0]:
silver_df=spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")

In [0]:
silver_df.display()

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-07-03T14:49:09.907Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-07-03T14:49:09.907Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-07-03T14:49:09.907Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-07-03T14:49:09.907Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-07-03T14:49:09.907Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-07-03T14:49:09.907Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-07-03T14:49:09.907Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-07-03T14:49:09.907Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-07-03T14:49:09.907Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-07-03T14:49:09.907Z,products.csv,1388


In [0]:
print("before dropping duplicates",silver_df.count())
silver_df=silver_df.dropDuplicates(["product_id"]) 
print("after dropping duplicates",silver_df.count())

before dropping duplicates 20
after dropping duplicates 18


In [0]:
silver_df.select("product_name").distinct().display()

product_name
SportsBar Energy Bar Choco Fudge (60g)
SportsBar Energy Bar Choco Fudge (40g)
SportsBar Energy Bar Choco Fudge (25g)
SportsBar Protien Bar Peanut Crunch (45g)
SportsBar Protien Bar Peanut Crunch (55g)
SportsBar Protien Bar Peanut Crunch (65g)
SportsBar Granola Crunch Honey Almond (400g)
SportsBar Granola Crunch Honey Almond (300g)
SportsBar Granola Crunch Honey Almond (200g)
SportsBar Greek Yogurt Pro Vanilla (200g)


In [0]:
silver_df=silver_df.withColumn("category",when(col("category").isNull() , None) .otherwise(initcap("category")))

In [0]:
silver_df.select("category").distinct().display()

category
Energy Bars
Protien Bars
Granola & Cereals
Recovery Dairy
Healthy Snacks
Electrolyte Mix


In [0]:
# Replacing protien with Protein
silver_df=silver_df.withColumn("product_name",regexp_replace(col("product_name"),"(?i)protien","Protein"))

In [0]:
silver_df.select("category").distinct().display()

category
Energy Bars
Protien Bars
Granola & Cereals
Recovery Dairy
Healthy Snacks
Electrolyte Mix


In [0]:
silver_df=silver_df.withColumn("division",
    when(col("category")=="Protien Bars","Nutrition Bars")\
    .when(col("category")=="Energy Bars","Nutrition Bars")\
    .when(col("category")=="Granola & Cereals","Breakfast Foods")\
    .when(col("category")=="Recovery Dairy","Dairy and Recovery")\
    .when(col("category")=="Healthy Snacks","Healthy Snacks")\
    .when(col("category")=="Electrolyte Mix","Hydrations")
) 


In [0]:
#creating variations by extracting 
# SportsBar Energy Bar Choco Fudge (60g) --> 60g
#SportsBar Oats Cookie Bites ChocoChip (500g) --> 500g
silver_df=silver_df.withColumn("variations",regexp_extract(col("product_name"),r"\(([^)]+)\)",1))

In [0]:
#product_id is not reliable so creating product_code using sha2 fucntion based on product_name
silver_df=silver_df.withColumn("product_code",sha2(col("product_name").cast("string"),256))\
    .withColumn("product_id",when(col("product_id").cast("string").rlike("^[0-9]+$"),col("product_id").cast("string"))\
    .otherwise(lit("9999999").cast("string"))) 

In [0]:
silver_df.display()

product_name,product_id,category,read_timestamp,file_name,file_size,division,variations,product_code
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,60g,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,40g,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,25g,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
SportsBar Protein Bar Peanut Crunch (45g),25891201,Protien Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,45g,2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protien Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,55g,0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28
SportsBar Protein Bar Peanut Crunch (65g),25891203,Protien Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,65g,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-07-03T14:49:09.907Z,products.csv,1388,Breakfast Foods,400g,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-07-03T14:49:09.907Z,products.csv,1388,Breakfast Foods,300g,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-07-03T14:49:09.907Z,products.csv,1388,Breakfast Foods,200g,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2026-07-03T14:49:09.907Z,products.csv,1388,Dairy and Recovery,200g,da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05


In [0]:
silver_df.display()

product_name,product_id,category,read_timestamp,file_name,file_size,division,variations,product_code
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,60g,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,40g,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,25g,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
SportsBar Protein Bar Peanut Crunch (45g),25891201,Protien Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,45g,2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protien Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,55g,0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28
SportsBar Protein Bar Peanut Crunch (65g),25891203,Protien Bars,2026-07-03T14:49:09.907Z,products.csv,1388,Nutrition Bars,65g,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-07-03T14:49:09.907Z,products.csv,1388,Breakfast Foods,400g,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-07-03T14:49:09.907Z,products.csv,1388,Breakfast Foods,300g,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-07-03T14:49:09.907Z,products.csv,1388,Breakfast Foods,200g,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2026-07-03T14:49:09.907Z,products.csv,1388,Dairy and Recovery,200g,da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05


In [0]:
 silver_df.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed',True)\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

# **Gold**

In [0]:
gold_source=spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")

In [0]:
gold_source=gold_source.select("product_name","product_id","category","division","variations","product_code")

In [0]:
gold_source.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed',True)\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
gold=spark.sql(f"select * from {catalog}.{gold_schema}.sb_dim_{data_source}")
display(gold)

product_name,product_id,category,division,variations,product_code
SportsBar Oats Cookie Bites ChocoChip (180g),25891503,Healthy Snacks,Healthy Snacks,180g,062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,Nutrition Bars,40g,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,Breakfast Foods,300g,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
SportsBar Greek Yogurt Pro Vanilla (80g),25891403,Recovery Dairy,Dairy and Recovery,80g,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,Breakfast Foods,400g,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
SportsBar Oats Cookie Bites ChocoChip (350g),9999999,Healthy Snacks,Healthy Snacks,350g,5931334e4cbe6b3792c209e8394e87aa21b83816b47d99375e4ff25e651ce63a
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protien Bars,Nutrition Bars,55g,0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28
SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),25891601,Electrolyte Mix,Hydrations,30 Sachets,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742
SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),25891602,Electrolyte Mix,Hydrations,15 Sachets,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,Breakfast Foods,200g,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f


In [0]:
parent_table=DeltaTable.forName(spark,"fmcg.gold.dim_products")
child_table=spark.sql(f"select product_code,variations as variant,division,category,product_name as product from fmcg.gold.sb_dim_products")

In [0]:
#upserting into parent table
parent_table.alias("target").merge(
    source=child_table.alias("source"),
    condition="target.product_code=source.product_code")\
    .whenMatchedUpdate(
        set={
        "target.variant":"source.variant" ,
        "target.division":"source.division",
        "target.category":"source.category",
        "target.product":"source.product" 
        }
    )\
    .whenNotMatchedInsert(
       values={
        "target.product_code":"source.product_code",
       "target.variant":"source.variant" ,
        "target.division":"source.division",
        "target.category":"source.category",
        "target.product":"source.product"  
       }
        
    ).execute()
 

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from fmcg.gold.dim_products;

product_code,division,category,product,variant
ARCHDDE20D,Archery,Arrows,PX Carbon Arrow Set,12 Pack
ARCH158F41,Archery,Finger Tab,LX Leather Finger Tab,Universal
ARCHAFF0E4,Archery,Bow,AX Precision Recurve Bow,26 lbs
ARCH6B94F7,Archery,Bow,AX Precision Recurve Bow,30 lbs
ARCH5D1FE7,Archery,Bow Stringer,BX Bow Stringing Tool,Standard
ARCH7B49A9,Archery,Arm Guard,NX Archery Arm Guard,Medium
ARCH497D34,Archery,Target,TX Foam Archery Target,80 cm
ARCHE71D79,Archery,Arm Guard,NX Archery Arm Guard,Large
ARCHDD8749,Archery,Target,TX Foam Archery Target,60 cm
BADMC045D4,Badminton,Badminton Racket,BX AeroLite Badminton Racket,Head Heavy
